In [2]:
# =========================================================
# AI QR CODE REVIVER + RISK ANALYZER
# =========================================================
# =========================================================
# STEP 1
# =========================================================
!apt-get install -y libzbar0
!pip install opencv-python pyzbar gradio pillow tensorflow
import cv2
import numpy as np
import gradio as gr

# =========================================================
# STEP 2
# =========================================================

from PIL import Image
from urllib.parse import urlparse

# TensorFlow
import tensorflow as tf

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    UpSampling2D,
    concatenate
)

print("Libraries Loaded")


# =========================================================
# STEP 3 U-NET
# =========================================================

def build_unet():

    inputs = Input((256,256,1))

    # =========================
    # ENCODER
    # =========================

    c1 = Conv2D(
        16,
        3,
        activation='relu',
        padding='same'
    )(inputs)

    c1 = Conv2D(
        16,
        3,
        activation='relu',
        padding='same'
    )(c1)

    p1 = MaxPooling2D((2,2))(c1)

    c2 = Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(p1)

    c2 = Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(c2)

    p2 = MaxPooling2D((2,2))(c2)

    # =========================
    # BOTTLENECK
    # =========================

    c3 = Conv2D(
        64,
        3,
        activation='relu',
        padding='same'
    )(p2)

    c3 = Conv2D(
        64,
        3,
        activation='relu',
        padding='same'
    )(c3)

    # =========================
    # DECODER
    # =========================

    u1 = UpSampling2D((2,2))(c3)

    u1 = concatenate([u1, c2])

    c4 = Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(u1)

    c4 = Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(c4)

    u2 = UpSampling2D((2,2))(c4)

    u2 = concatenate([u2, c1])

    c5 = Conv2D(
        16,
        3,
        activation='relu',
        padding='same'
    )(u2)

    c5 = Conv2D(
        16,
        3,
        activation='relu',
        padding='same'
    )(c5)

    outputs = Conv2D(
        1,
        1,
        activation='sigmoid'
    )(c5)

    model = Model(
        inputs=[inputs],
        outputs=[outputs]
    )

    return model


# Build model
unet_model = build_unet()

print("U-Net Loaded")


# =========================================================
# STEP 4 — REMOVE COLORED TAMPERING
# =========================================================

def remove_colored_noise(image):

    hsv = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2HSV
    )

    # Yellow mask
    lower_yellow = np.array([15, 80, 80])
    upper_yellow = np.array([40, 255, 255])

    yellow_mask = cv2.inRange(
        hsv,
        lower_yellow,
        upper_yellow
    )

    # Green mask
    lower_green = np.array([35, 50, 50])
    upper_green = np.array([90, 255, 255])

    green_mask = cv2.inRange(
        hsv,
        lower_green,
        upper_green
    )

    # Combine masks
    mask = cv2.bitwise_or(
        yellow_mask,
        green_mask
    )

    # Inpaint
    cleaned = cv2.inpaint(
        image,
        mask,
        3,
        cv2.INPAINT_TELEA
    )

    return cleaned


# =========================================================
# STEP 5 — UPSCALE IMAGE
# =========================================================
# Helps dense/payment QR decoding
# =========================================================

def upscale_image(image):

    scaled = cv2.resize(
        image,
        None,
        fx=3,
        fy=3,
        interpolation=cv2.INTER_CUBIC
    )

    return scaled


# =========================================================
# STEP 6 — OPENCV QR DECODER
# =========================================================

def decode_qr(image):

    detector = cv2.QRCodeDetector()

    data, points, _ = detector.detectAndDecode(image)

    if data:
        return data

    return None


# =========================================================
# STEP 7 — MULTIPLE ENHANCEMENT METHODS
# =========================================================

def enhancement_variants(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    variants = []

    # =====================================
    # 1. DENOISE + SHARPEN
    # =====================================

    denoised = cv2.fastNlMeansDenoising(gray)

    kernel = np.array([
        [0,-1,0],
        [-1,5,-1],
        [0,-1,0]
    ])

    sharpened = cv2.filter2D(
        denoised,
        -1,
        kernel
    )

    variants.append(sharpened)

    # =====================================
    # 2. ADAPTIVE THRESHOLD
    # =====================================

    adaptive = cv2.adaptiveThreshold(
        sharpened,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    variants.append(adaptive)

    # =====================================
    # 3. OTSU THRESHOLD
    # =====================================

    _, otsu = cv2.threshold(
        sharpened,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    variants.append(otsu)

    # =====================================
    # 4. CLAHE CONTRAST
    # =====================================

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    contrast = clahe.apply(gray)

    variants.append(contrast)

    return variants


# =========================================================
# STEP 8 — U-NET QR RECOVERY
# =========================================================

def unet_qr_recovery(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    resized = cv2.resize(
        gray,
        (256,256)
    )

    normalized = resized / 255.0

    input_tensor = np.expand_dims(
        normalized,
        axis=(0,-1)
    )

    prediction = unet_model.predict(
        input_tensor,
        verbose=0
    )

    recovered = prediction[0,:,:,0]

    recovered = (recovered * 255).astype(np.uint8)

    recovered = cv2.resize(
        recovered,
        (gray.shape[1], gray.shape[0])
    )

    recovered = cv2.threshold(
        recovered,
        127,
        255,
        cv2.THRESH_BINARY
    )[1]

    return recovered


# =========================================================
# STEP 9 — SMART MULTI-STAGE QR RECOVERY
# =========================================================

def revive_qr(image):

    # =====================================
    # ATTEMPT 1 — ORIGINAL
    # =====================================

    result = decode_qr(image)

    if result:
        return result, "Decoded Normally"

    # =====================================
    # ATTEMPT 2 — UPSCALE
    # =====================================

    upscaled = upscale_image(image)

    result = decode_qr(upscaled)

    if result:
        return result, "Recovered After Upscaling"

    # =====================================
    # ATTEMPT 3 — COLOR CLEANUP
    # =====================================

    cleaned = remove_colored_noise(upscaled)

    result = decode_qr(cleaned)

    if result:
        return result, "Recovered After Color Cleanup"

    # =====================================
    # ATTEMPT 4 — MULTI ENHANCEMENT
    # =====================================

    variants = enhancement_variants(cleaned)

    for idx, variant in enumerate(variants):

        result = decode_qr(variant)

        if result:

            return result, f"Recovered Using Enhancement Variant {idx+1}"

    # =====================================
    # ATTEMPT 5 — U-NET AI
    # =====================================

    recovered = unet_qr_recovery(cleaned)

    result = decode_qr(recovered)

    if result:
        return result, "Recovered Using U-Net AI"

    return None, "Failed to Decode"


# =========================================================
# STEP 10 — URL RISK ANALYZER
# =========================================================

def analyze_url_risk(data):

    # Non URL
    if not data.startswith("http"):

        return "Safe", "Non-URL QR"

    parsed = urlparse(data)

    domain = parsed.netloc.lower()

    suspicious_keywords = [
        "login",
        "verify",
        "secure",
        "bank",
        "wallet",
        "bonus",
        "gift",
        "crypto",
        "password"
    ]

    dangerous_patterns = [
        ".ru",
        ".tk",
        ".xyz"
    ]

    # Dangerous check
    for pattern in dangerous_patterns:

        if pattern in domain:

            return "Dangerous", f"Risky domain detected: {pattern}"

    # Suspicious keyword check
    for keyword in suspicious_keywords:

        if keyword in data.lower():

            return "Suspicious", f"Keyword detected: {keyword}"

    # Long URL
    if len(data) > 120:

        return "Suspicious", "Very long URL"

    return "Safe", "No major threats detected"


# =========================================================
# STEP 11 — MAIN PIPELINE
# =========================================================

def process_qr(image):

    # PIL → NumPy
    image = np.array(image)

    # RGB → BGR
    image = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2BGR
    )

    # Recover QR
    decoded_data, status = revive_qr(image)

    # Failed
    if not decoded_data:

        return {
            "Status": status,
            "Decoded Data": None,
            "Risk Level": "Unknown",
            "Reason": "QR could not be decoded"
        }

    # Risk analysis
    risk, reason = analyze_url_risk(decoded_data)

    return {
        "Status": status,
        "Decoded Data": decoded_data,
        "Risk Level": risk,
        "Reason": reason
    }


# =========================================================
# STEP 12 — GRADIO UI
# =========================================================

interface = gr.Interface(
    fn=process_qr,

    inputs=gr.Image(type="pil"),

    outputs=gr.JSON(),

    title="AI QR Code Reviver and Risk Analyzer",

    description="""
    Upload blurry, damaged, or tampered QR images.

    Features:
    - Multi-stage QR recovery
    - Payment QR support improvement
    - Upscaling for dense QR
    - Color tampering cleanup
    - Multiple enhancement retries
    - U-Net AI restoration
    - URL risk analysis
    """
)

# Launch app
interface.launch(share=True)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libzbar0 is already the newest version (0.23.92-4build2).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Libraries Loaded
U-Net Loaded
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://98af552957df2e2035.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
